In [ ]:
import os
import sys

# Get the path where this notebook is running
notebook_dir = os.getcwd()

# Navigate up to the 'ai_model' directory
ai_model_dir = os.path.abspath(os.path.join(notebook_dir, ".."))

# Navigate up to the project root directory
project_root = os.path.abspath(os.path.join(ai_model_dir, ".."))

# Add both paths to Python's module search path
for path in [ai_model_dir, project_root]:
    if path not in sys.path:
        sys.path.insert(0, path)

# Import preprocessing functions
try:
    from src.preprocess import clean_text, extract_red_flags
    print("✅ Success! Imported from src.preprocess")
except ModuleNotFoundError:
    from AI_model.src.preprocess import clean_text, extract_red_flags
    print("✅ Success! Imported from ai_model.src.preprocess")

✅ Success! Imported from src.preprocess


In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Resolve search paths for imports
if 'ai_model_dir' in globals():
    for path_dir in [ai_model_dir, os.path.join(ai_model_dir, "src")]:
        if os.path.exists(path_dir) and path_dir not in sys.path:
            sys.path.insert(0, path_dir)

try:
    from src.preprocess import clean_text
except ModuleNotFoundError:
    from src.preprocess import clean_text

# Path to the Kaggle CSV dataset
dataset_path = os.path.join(ai_model_dir, "data", "raw", "fake_real_job_postings_3000x25.csv")

if os.path.exists(dataset_path):
    print(f"✅ Found dataset! Loading from: {dataset_path}")
    df = pd.read_csv(dataset_path)
    
    # Combine available text columns into full_text
    possible_cols = ['title', 'company_profile', 'description', 'requirements', 'benefits', 'text']
    text_cols = [c for c in possible_cols if c in df.columns]
    
    for col in text_cols:
        df[col] = df[col].fillna('')
        
    df['full_text'] = df[text_cols].apply(lambda row: ' '.join(row), axis=1)
    
    # Identify target label column
    target_col = 'fraudulent' if 'fraudulent' in df.columns else ('label' if 'label' in df.columns else df.columns[-1])
    
    X_raw = df['full_text'].tolist()
    
    # Convert target y into clean integer 0s and 1s
    raw_y = df[target_col].fillna(0).tolist()
    y = np.array([1 if str(val).strip().lower() in ['1', '1.0', 'true', 'fraudulent', 'fake', 'scam'] else 0 for val in raw_y], dtype=int)
    
    print(f"Total real samples loaded: {len(X_raw)}")
    print(f"Class distribution -> Legitimate (0): {sum(y == 0)}, Scam (1): {sum(y == 1)}")
else:
    print(f"⚠️ Dataset NOT found at: {dataset_path}")
    X_raw, y = [], np.array([])

# Preprocess all raw text samples
X_clean = [clean_text(text) for text in X_raw]
print("✅ Text preprocessing and label encoding completed successfully!")

✅ Found dataset! Loading from: c:\Users\geeth\OneDrive\Desktop\FrontEndProjects\TrustIntern-AI\AI_model\data\raw\fake_real_job_postings_3000x25.csv
Total real samples loaded: 3000
Class distribution -> Legitimate (0): 3000, Scam (1): 0
✅ Text preprocessing and label encoding completed successfully!


In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier

print("Vectorizing 3,000 preprocessed texts...")
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, stop_words='english')
X_vec = vectorizer.fit_transform(X_clean)

# Calculate ratio of legitimate (0) to scam (1) samples
num_legit = sum(y == 0)
num_scams = sum(y == 1)
scale_weight = float(num_legit) / max(num_scams, 1)

print(f"Dataset Ratio -> Legit: {num_legit}, Scams: {num_scams} (scale_pos_weight: {scale_weight:.2f})")

print("Training Weighted XGBoost Model...")
model = XGBClassifier(
    n_estimators=100, 
    max_depth=5, 
    learning_rate=0.1, 
    scale_pos_weight=scale_weight,  # Penalizes missing scams
    random_state=42
)
model.fit(X_vec, y)

print("✅ Model training completed successfully with class balancing!")

Vectorizing 3,000 preprocessed texts...
Dataset Ratio -> Legit: 3000, Scams: 0 (scale_pos_weight: 3000.00)
Training Weighted XGBoost Model...
✅ Model training completed successfully with class balancing!


In [36]:
import os
import joblib

models_dir = os.path.join(ai_model_dir, "models")
os.makedirs(models_dir, exist_ok=True)
model_path = os.path.join(models_dir, "scam_detector.pkl")

joblib.dump({'vectorizer': vectorizer, 'model': model}, model_path)
print(f"🎉 SUCCESS! Model saved to: {model_path}")

🎉 SUCCESS! Model saved to: c:\Users\geeth\OneDrive\Desktop\FrontEndProjects\TrustIntern-AI\AI_model\models\scam_detector.pkl
